In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
orders_df = spark.read.parquet("data/external_table/part-00000-6109de51-a0e8-41af-839d-58e5ef769898-c000.snappy.parquet", inferSchema=True, header=True)

In [3]:
orders_df.createOrReplaceTempView('orders')

### 4. Top 10 customers with most orders placed

In [4]:
orders_df.show(10)

+--------+-------------------+-----------+---------------+
|order_id|         order_date|customer_id|   order_status|
+--------+-------------------+-----------+---------------+
|       1|2013-07-25 00:00:00|      11599|         CLOSED|
|       2|2013-07-25 00:00:00|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|      12111|       COMPLETE|
|       4|2013-07-25 00:00:00|       8827|         CLOSED|
|       5|2013-07-25 00:00:00|      11318|       COMPLETE|
|       6|2013-07-25 00:00:00|       7130|       COMPLETE|
|       7|2013-07-25 00:00:00|       4530|       COMPLETE|
|       8|2013-07-25 00:00:00|       2911|     PROCESSING|
|       9|2013-07-25 00:00:00|       5657|PENDING_PAYMENT|
|      10|2013-07-25 00:00:00|       5648|PENDING_PAYMENT|
+--------+-------------------+-----------+---------------+
only showing top 10 rows



In [5]:
spark.sql("""
            SELECT customer_id, COUNT(order_id) as order_count
            FROM orders
            GROUP BY customer_id
            ORDER BY order_count DESC
""")

customer_id,order_count
5897,16
569,16
12431,16
6316,16
5654,15
5624,15
12284,15
221,15
4320,15
5283,15


In [6]:
from pyspark.sql.functions import count
orders_df.groupBy("customer_id").agg(count('order_id').alias("order_count")).sort('order_count', ascending=False)

customer_id,order_count
5897,16
569,16
12431,16
6316,16
5654,15
5624,15
12284,15
221,15
4320,15
5283,15


In [ ]:
spark.stop()